In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv("../ocr-result1/master_summary.csv", index_col=False)

In [3]:
df['province'] = "ลำปาง"

In [4]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
sd2d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                            sd2d[unit['subdistrict']] = unit['district']
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
print(sd2d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
{'บ้านแลง': 'เมืองลำปาง', 'บ้านเสด็จ': 'เมืองลำปาง', 'หลวงเหนือ': 'งาว', 'หลวงใต้': 'งาว', 'บ้านโป่ง': 'งาว', 'บ้านร้อง': 'งาว', 'ปงเตา': 'งาว', 'นาแก': 'งาว', 'บ้านอ้อน': 'งาว', 'บ้านแหง': 'งาว', 'บ้านหวด': 'งาว', 'แม่ตีบ': 'งาว', 'แจ้ห่ม': 'แจ้ห่ม', 'บ้านสา': 'แจ้ห่ม', 'ปงดอน': 'แจ้ห่ม', 'แม่สุก': 'แจ้ห่ม', 'เมืองมาย': 'แจ้ห่ม', 'ทุ่งผึ้ง': 'แจ้ห่ม', 'วิเชตนคร': 'แจ้ห่ม', 'ทุ่งฮั้ว': 'วังเหนือ', 'วังเหนือ': 'วังเหนือ', 'วังใต้': 'วังเหนือ', 'ร่องเคาะ': 'วังเ

In [5]:
d['นอกเขต'] = dict()
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    sd2d[sub_dist] = 'นอกเขต'
    d['นอกเขต'][sub_dist] = 1
    # all_subdistrict.append(sub_dist)

In [6]:
d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)
sd2d['บ้านใหม่'] = 'วังเหนือ'
sd2d['แจ้ห่ม(ในเขต)'] = "แจ้ห่ม"
sd2d['แจ้ห่ม(นอกเขต)'] = "แจ้ห่ม"

In [7]:
all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

In [8]:
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    all_subdistrict.append(sub_dist)

In [9]:
wrong_districts = []
for wrong_district in df['district'].unique().tolist():
    if wrong_district not in all_distrcit and wrong_district != 'นอกเขต':
        wrong_districts.append(wrong_district)

wrong_districts

['อำเภองาว',
 'อำเภอวังเหนือ',
 'อำเภอเมืองปาน',
 'อำเภอเมืองลำปาง (ต.บ้านแลง ต.บ้านเสด็จ)',
 'อำเภอแจ้ห่ม']

In [10]:
from rapidfuzz import process
def automate_edit(feature, check, wrongs, df):
    wrong2correct = dict()
    scores = dict()
    for text in wrongs:
        match = process.extractOne(text, check, score_cutoff=50)

        if match:
            result, score, index = match
            wrong2correct[text] = result
            scores[text] = round(score,2)
            if scores[text] >= 0.8:
                df.loc[df[feature] == text, feature] = result
            print(f"OCR: '{text}' -> Corrected: '{result}' (Confidence: {score:.2f}%)")
        else:
            wrong2correct[text] = "can't find"
            scores[text] = 0
            print(f"OCR: '{text}' -> [Manual Review Needed]")
    return wrong2correct, scores

In [11]:
df_test = df.copy()

In [12]:
wrong2correct, scores = automate_edit('district', all_distrcit, wrong_districts, df_test)

OCR: 'อำเภองาว' -> Corrected: 'งาว' (Confidence: 90.00%)
OCR: 'อำเภอวังเหนือ' -> Corrected: 'วังเหนือ' (Confidence: 90.00%)
OCR: 'อำเภอเมืองปาน' -> Corrected: 'เมืองปาน' (Confidence: 90.00%)
OCR: 'อำเภอเมืองลำปาง (ต.บ้านแลง ต.บ้านเสด็จ)' -> Corrected: 'เมืองลำปาง' (Confidence: 90.00%)
OCR: 'อำเภอแจ้ห่ม' -> Corrected: 'แจ้ห่ม' (Confidence: 90.00%)


In [13]:
df_test['district'].unique().tolist()

['นอกเขต', 'งาว', 'วังเหนือ', 'เมืองปาน', 'เมืองลำปาง', 'แจ้ห่ม']

In [14]:
df_test['sub-district'].unique().tolist()

['ชุดที่ 1',
 'ชุดที่ 10',
 'ชุดที่ 11',
 'ชุดที่ 12',
 'ชุดที่ 13',
 'ชุดที่ 2',
 'ชุดที่ 3',
 'ชุดที่ 4',
 'ชุดที่ 5',
 'ชุดที่ 6',
 'ชุดที่ 7',
 'ชุดที่ 8',
 'ชุดที่ 9',
 'นาแก',
 'บ้านร้อง',
 'บ้านหวด',
 'บ้านอ้อน',
 'บ้านแหง',
 'บ้านโป่ง',
 'ปงเตา',
 'หลวงเหนือ',
 'หลวงใต้',
 'แม่ตีบ',
 'วังใต้',
 'ทุ่งฮั้ว',
 'บ้านใหม่',
 'ร่องเคาะ',
 'วังซ้าย',
 'วังทรายคำ',
 'วังทอง',
 'วังเหนือ',
 'วังแก้ว',
 'บ้านขอ',
 'ทุ่งกว๋าว',
 'หัวเมือง',
 'เมืองปาน',
 'เเจ้ซ้อน',
 'แจ้ซ้อน',
 'บ้านเสด็จ',
 'บ้านแลง',
 'ทุ่งผึ้ง',
 'บ้านสา',
 'วิเชตนคร',
 'เมืองมาย',
 'แม่สุก',
 'แจ้ห่ม(ในเขต)',
 'แจ้ห่ม(นอกเขต)',
 'ปงดอน']

In [15]:
wrong_sds = []
for wrong_sd in df_test['sub-district'].unique().tolist():
    if wrong_sd not in all_subdistrict:
        wrong_sds.append(wrong_sd)

wrong_sds

['เเจ้ซ้อน']

In [16]:
df_test.loc[df_test['sub-district'] == 'เเจ้ซ้อน', 'sub-district'] = 'แจ้ซ้อน'

In [17]:
wrong_sds = []
for wrong_sd in df_test['sub-district'].unique().tolist():
    if wrong_sd not in all_subdistrict:
        wrong_sds.append(wrong_sd)

wrong_sds

[]

In [18]:
l = df_test['sub-district'].unique().tolist()
for sd in all_subdistrict:
    if sd not in l:
        print(f"เขต {sd} หายไปทั้ง บช และ เขต")

In [19]:
df = df_test

In [20]:
# เปลี่ยนชื่อตัวแปร type เป็น type_val เพื่อไม่ให้ซ้ำกับคำสั่ง type() ของ Python
for sd in all_subdistrict:
    for type_val in ['เขต', 'บช']:
        # 1. ดึงอำเภอ (dist) และจำนวนหน่วยที่คาดหวังทั้งหมด
        dist = sd2d[sd] 
        expected_count = d[dist][sd]
        if dist == 'นอกเขต': break
        
        expected_units = set(range(1, expected_count + 1))
        
        # 2. ดึงข้อมูล unit_number ของตำบล/ประเภทนี้ มาเป็น Pandas Series
        unit_series = df[(df['sub-district'] == sd) & (df['type'] == type_val)]['unit_number'].astype(int)
        
        # 3. สร้าง Set สำหรับเช็ค ขาด/เลขแปลกปลอม
        actual_units_set = set(unit_series.unique())
        
        # 4. ใช้ value_counts() เพื่อนับว่าแต่ละหน่วยมาโผล่มากี่แถว
        counts = unit_series.value_counts()
        
        # กรองเอาเฉพาะ index (เลขหน่วย) ที่มี count > 1 (แปลว่ามาซ้ำ)
        duplicates = sorted(counts[counts > 1].index.tolist())
        
        # 5. หา Set Difference ลบกันเพื่อหาส่วนต่าง
        missing = sorted(list(expected_units - actual_units_set)) # ควรมีแต่ไม่มี
        extra = sorted(list(actual_units_set - expected_units))   # ไม่ควรมีแต่ดันมี (OCR อ่านเลขผิด)
        
        # 6. ถ้ามีความผิดปกติอย่างใดอย่างหนึ่ง ให้ Print แจ้งเตือน
        if missing or extra or duplicates:
            print(f"\n{type_val}, ลำปาง, {dist}, {sd}")
            print(f"  > จำนวนแถว: มี {len(unit_series)} แถว จากที่ควรมี {expected_count} หน่วย")
            
            if missing:
                print(f"  > ❌ หน่วยที่หายไป: {missing}")
            if extra:
                print(f"  > ⚠️ หน่วยแปลกปลอม (อ่านเลขผิด): {extra}")
            if duplicates:
                print(f"  > 🔄 หน่วยที่ซ้ำ (มีมากกว่า 1 แถว): {duplicates}")
                # ปริ้นบอกด้วยว่าหน่วยที่ซ้ำนั้น ซ้ำกี่รอบ
                for dup in duplicates:
                    print(f"      - หน่วยที่ {dup} มา {counts[dup]} รอบ")


เขต, ลำปาง, เมืองลำปาง, บ้านเสด็จ
  > จำนวนแถว: มี 18 แถว จากที่ควรมี 19 หน่วย
  > ❌ หน่วยที่หายไป: [19]

เขต, ลำปาง, งาว, บ้านโป่ง
  > จำนวนแถว: มี 10 แถว จากที่ควรมี 12 หน่วย
  > ❌ หน่วยที่หายไป: [9, 11]

บช, ลำปาง, งาว, บ้านโป่ง
  > จำนวนแถว: มี 11 แถว จากที่ควรมี 12 หน่วย
  > ❌ หน่วยที่หายไป: [9]

เขต, ลำปาง, วังเหนือ, ทุ่งฮั้ว
  > จำนวนแถว: มี 11 แถว จากที่ควรมี 12 หน่วย
  > ❌ หน่วยที่หายไป: [10]


Mannual add

In [21]:
df.loc[len(df)] = ["เขต", "ลำปาง", "เมืองลำปาง", "บ้านเสด็จ", 19,284,200]
df.loc[len(df)] = ["เขต", "ลำปาง", "งาว", "บ้านโป่ง", 9,448,292]
df.loc[len(df)] = ["เขต", "ลำปาง", "งาว", "บ้านโป่ง", 11,396,218]
df.loc[len(df)] = ["บช", "ลำปาง", "งาว", "บ้านโป่ง", 9,448,292]
df.loc[len(df)] = ["เขต", "ลำปาง", "วังเหนือ", "ทุ่งฮั้ว", 10,-1,-1]

In [22]:
# เปลี่ยนชื่อตัวแปร type เป็น type_val เพื่อไม่ให้ซ้ำกับคำสั่ง type() ของ Python
for sd in all_subdistrict:
    for type_val in ['เขต', 'บช']:
        # 1. ดึงอำเภอ (dist) และจำนวนหน่วยที่คาดหวังทั้งหมด
        dist = sd2d[sd] 
        expected_count = d[dist][sd]
        if dist == 'นอกเขต': break
        
        expected_units = set(range(1, expected_count + 1))
        
        # 2. ดึงข้อมูล unit_number ของตำบล/ประเภทนี้ มาเป็น Pandas Series
        unit_series = df[(df['sub-district'] == sd) & (df['type'] == type_val)]['unit_number'].astype(int)
        
        # 3. สร้าง Set สำหรับเช็ค ขาด/เลขแปลกปลอม
        actual_units_set = set(unit_series.unique())
        
        # 4. ใช้ value_counts() เพื่อนับว่าแต่ละหน่วยมาโผล่มากี่แถว
        counts = unit_series.value_counts()
        
        # กรองเอาเฉพาะ index (เลขหน่วย) ที่มี count > 1 (แปลว่ามาซ้ำ)
        duplicates = sorted(counts[counts > 1].index.tolist())
        
        # 5. หา Set Difference ลบกันเพื่อหาส่วนต่าง
        missing = sorted(list(expected_units - actual_units_set)) # ควรมีแต่ไม่มี
        extra = sorted(list(actual_units_set - expected_units))   # ไม่ควรมีแต่ดันมี (OCR อ่านเลขผิด)
        
        # 6. ถ้ามีความผิดปกติอย่างใดอย่างหนึ่ง ให้ Print แจ้งเตือน
        if missing or extra or duplicates:
            print(f"\n{type_val}, ลำปาง, {dist}, {sd}")
            print(f"  > จำนวนแถว: มี {len(unit_series)} แถว จากที่ควรมี {expected_count} หน่วย")
            
            if missing:
                print(f"  > ❌ หน่วยที่หายไป: {missing}")
            if extra:
                print(f"  > ⚠️ หน่วยแปลกปลอม (อ่านเลขผิด): {extra}")
            if duplicates:
                print(f"  > 🔄 หน่วยที่ซ้ำ (มีมากกว่า 1 แถว): {duplicates}")
                # ปริ้นบอกด้วยว่าหน่วยที่ซ้ำนั้น ซ้ำกี่รอบ
                for dup in duplicates:
                    print(f"      - หน่วยที่ {dup} มา {counts[dup]} รอบ")

In [23]:
cols_to_check = ['จำนวนผู้มีสิทธิมาเลือกตั้ง', 'จำนวนผู้มาแสดงตน']
for col in cols_to_check:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1)

df_khet = df[df['type'] == 'เขต'].copy()
df_list = df[df['type'] == 'บช'].copy()

# 3. นำ 2 ตารางมา Join กันด้วย Key สถานที่ (ตำบล และ หน่วยที่)
# suffixes จะช่วยเติมท้ายชื่อคอลัมน์ที่ซ้ำกันให้รู้ว่ามาจากตารางไหน
merged_df = pd.merge(
    df_khet, 
    df_list, 
    on=['province', 'district', 'sub-district', 'unit_number'], 
    how='inner', 
    suffixes=('_เขต', '_บช')
)

# 4. สร้างเงื่อนไขตรวจสอบ (Mismatch Conditions)
# เงื่อนไข 1: จำนวนผู้มีสิทธิฯ ไม่เท่ากัน
cond_eligible = merged_df['จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต'] != merged_df['จำนวนผู้มีสิทธิมาเลือกตั้ง_บช']

# เงื่อนไข 2: จำนวนผู้มาแสดงตน ไม่เท่ากัน
cond_voted = merged_df['จำนวนผู้มาแสดงตน_เขต'] != merged_df['จำนวนผู้มาแสดงตน_บช']

# 5. กรองเอาเฉพาะแถวที่ผิดปกติ (ตรงเงื่อนไข 1 หรือ 2 อย่างใดอย่างหนึ่ง)
error_df = merged_df[cond_eligible | cond_voted].copy()

# แสดงผลลัพธ์
if error_df.empty:
    print("✅ ข้อมูลสมบูรณ์! ยอดผู้มีสิทธิและผู้มาแสดงตนของ 'เขต' และ 'บช' ตรงกันทุกหน่วย")
else:
    print(f"❌ พบหน่วยที่ข้อมูลไม่ตรงกันจำนวน {len(error_df)} หน่วย ดังนี้:")
    
    # เลือกเฉพาะคอลัมน์ที่จำเป็นมาดูเพื่อความสะอาดตา
    display_cols = [
        'district', 'sub-district', 'unit_number', 
        'จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต', 'จำนวนผู้มีสิทธิมาเลือกตั้ง_บช',
        'จำนวนผู้มาแสดงตน_เขต', 'จำนวนผู้มาแสดงตน_บช'
    ]
    

❌ พบหน่วยที่ข้อมูลไม่ตรงกันจำนวน 245 หน่วย ดังนี้:


In [24]:
error_df[display_cols]

,district,sub-district,unit_number,จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต,จำนวนผู้มีสิทธิมาเลือกตั้ง_บช,จำนวนผู้มาแสดงตน_เขต,จำนวนผู้มาแสดงตน_บช
13,งาว,นาแก,1,324,324,-1,214
14,งาว,นาแก,2,735,735,404,448
15,งาว,นาแก,3,37,37,244,249
16,งาว,นาแก,4,463,468,327,329
18,งาว,นาแก,6,277,297,192,192
...,...,...,...,...,...,...,...
348,แจ้ห่ม,ปงดอน,7,724,104,439,439
349,แจ้ห่ม,ปงดอน,8,300,300,20,104
350,แจ้ห่ม,ปงดอน,9,116,107,107,107
351,เมืองลำปาง,บ้านเสด็จ,19,284,451,200,282


In [25]:
df_compare = pd.read_csv("../master_summary.csv", index_col=0)
df_compare.head()

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,663,8,29,-1
1,เขต,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,605,32,63,-1
2,บช,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,668,8,24,-1
3,เขต,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,616,28,56,-1
4,บช,ลำปาง,นอกเขต,ชุดที่ 11,-1,700,664,6,30,-1


In [26]:
df_tset = df.copy()

In [27]:
import pandas as pd

# 1. ดึงคอลัมน์ที่ต้องใช้คำนวณจาก df_compare
cols_needed = ['sub-district', 'unit_number', 'valid_ballots', 'invalid_ballots', 'no_vote_ballots']
voter_mapping = df_compare[cols_needed].copy()

# ลบแถวซ้ำ โดยยึดตาม ตำบลและหน่วย (เพื่อให้เหลือแค่หน่วยละ 1 แถว)
voter_mapping = voter_mapping.drop_duplicates(subset=['sub-district', 'unit_number'])

# ทำการแปลงเป็นตัวเลขให้ชัวร์ก่อนบวกกัน (ถ้ามีตัวอักษรหลงมาจะกลายเป็น 0)
for col in ['valid_ballots', 'invalid_ballots', 'no_vote_ballots']:
    voter_mapping[col] = pd.to_numeric(voter_mapping[col], errors='coerce').fillna(0).astype(int)

# สร้างคอลัมน์ใหม่เพื่อเก็บผลรวม (บัตรดี + บัตรเสีย + ไม่ประสงค์ลงคะแนน)
voter_mapping['actual_voted'] = (
    voter_mapping['valid_ballots'] + 
    voter_mapping['invalid_ballots'] + 
    voter_mapping['no_vote_ballots']
)

# 2. ปรับ Data Type ของคีย์ให้ตรงกันก่อน Merge
voter_mapping['unit_number'] = voter_mapping['unit_number'].astype(int)
df['unit_number'] = df['unit_number'].astype(int)

# 3. Merge ข้อมูลผลรวมที่ได้ กลับเข้าไปใน df หลัก (ใช้แค่คีย์ และผลรวม)
df = df.merge(
    voter_mapping[['sub-district', 'unit_number', 'actual_voted']], 
    on=['sub-district', 'unit_number'], 
    how='left'
)

# 4. นำผลที่คำนวณได้ไปอัปเดตใส่คอลัมน์ "จำนวนผู้มาแสดงตน"
df['จำนวนผู้มาแสดงตน'] = df['actual_voted']

# 5. ลบคอลัมน์ชั่วคราวทิ้งเพื่อให้ตารางสะอาดเหมือนเดิม
df.drop(columns=['actual_voted'], inplace=True)

# ตรวจสอบผลลัพธ์
print(df.head())

  type province district sub-district  unit_number  \
0   บช    ลำปาง   นอกเขต     ชุดที่ 1           -1   
1  เขต    ลำปาง   นอกเขต     ชุดที่ 1           -1   
2   บช    ลำปาง   นอกเขต    ชุดที่ 10           -1   
3  เขต    ลำปาง   นอกเขต    ชุดที่ 10           -1   
4   บช    ลำปาง   นอกเขต    ชุดที่ 11           -1   

   จำนวนผู้มีสิทธิมาเลือกตั้ง  จำนวนผู้มาแสดงตน  
0                          -1               700  
1                          -1               700  
2                          -1               700  
3                          -1               700  
4                          -1               700  


In [28]:
cols_to_check = ['จำนวนผู้มีสิทธิมาเลือกตั้ง', 'จำนวนผู้มาแสดงตน']
for col in cols_to_check:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1)

df_khet = df[df['type'] == 'เขต'].copy()
df_list = df[df['type'] == 'บช'].copy()

# 3. นำ 2 ตารางมา Join กันด้วย Key สถานที่ (ตำบล และ หน่วยที่)
# suffixes จะช่วยเติมท้ายชื่อคอลัมน์ที่ซ้ำกันให้รู้ว่ามาจากตารางไหน
merged_df = pd.merge(
    df_khet, 
    df_list, 
    on=['province', 'district', 'sub-district', 'unit_number'], 
    how='inner', 
    suffixes=('_เขต', '_บช')
)

# 4. สร้างเงื่อนไขตรวจสอบ (Mismatch Conditions)
# เงื่อนไข 1: จำนวนผู้มีสิทธิฯ ไม่เท่ากัน
cond_eligible = merged_df['จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต'] != merged_df['จำนวนผู้มีสิทธิมาเลือกตั้ง_บช']

# เงื่อนไข 2: จำนวนผู้มาแสดงตน ไม่เท่ากัน
cond_voted = merged_df['จำนวนผู้มาแสดงตน_เขต'] != merged_df['จำนวนผู้มาแสดงตน_บช']

# 5. กรองเอาเฉพาะแถวที่ผิดปกติ (ตรงเงื่อนไข 1 หรือ 2 อย่างใดอย่างหนึ่ง)
error_df = merged_df[cond_eligible | cond_voted].copy()

# แสดงผลลัพธ์
if error_df.empty:
    print("✅ ข้อมูลสมบูรณ์! ยอดผู้มีสิทธิและผู้มาแสดงตนของ 'เขต' และ 'บช' ตรงกันทุกหน่วย")
else:
    print(f"❌ พบหน่วยที่ข้อมูลไม่ตรงกันจำนวน {len(error_df)} หน่วย ดังนี้:")
    
    # เลือกเฉพาะคอลัมน์ที่จำเป็นมาดูเพื่อความสะอาดตา
    display_cols = [
        'district', 'sub-district', 'unit_number', 
        'จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต', 'จำนวนผู้มีสิทธิมาเลือกตั้ง_บช',
        'จำนวนผู้มาแสดงตน_เขต', 'จำนวนผู้มาแสดงตน_บช'
    ]
    

❌ พบหน่วยที่ข้อมูลไม่ตรงกันจำนวน 193 หน่วย ดังนี้:


In [29]:
error_df[display_cols]

,district,sub-district,unit_number,จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต,จำนวนผู้มีสิทธิมาเลือกตั้ง_บช,จำนวนผู้มาแสดงตน_เขต,จำนวนผู้มาแสดงตน_บช
16,งาว,นาแก,4,463,468,329,329
18,งาว,นาแก,6,277,297,192,192
19,งาว,บ้านร้อง,1,440,442,316,316
20,งาว,บ้านร้อง,10,269,469,261,261
21,งาว,บ้านร้อง,11,305,-1,225,225
...,...,...,...,...,...,...,...
347,แจ้ห่ม,ปงดอน,6,80,84,56,56
348,แจ้ห่ม,ปงดอน,7,724,104,439,439
350,แจ้ห่ม,ปงดอน,9,116,107,107,107
351,เมืองลำปาง,บ้านเสด็จ,19,284,451,282,282


In [30]:
# ทุ่งฮั้ว มีการสลับ value ของหน่วยตั้งแต่ 1-12 ใหม่โดย1 ต้องเป็น 12 และ 2 เป็น 11 ไปเริ่อยๆจน 12 เป็น 1
df['unit_number'] = df['unit_number'].astype(int)

df = df.sort_values(by=['type', 'province', 'district', 'sub-district', 'unit_number'])

# 3. สร้างเงื่อนไขล็อกเป้า (Mask) เฉพาะตำบล "ทุ่งฮั้ว"
mask = df['sub-district'] == 'ทุ่งฮั้ว'

# 4. ทำการสลับค่าเฉพาะแถวที่เข้าเงื่อนไข (แยกสลับระหว่าง เขต และ บช)
# ใช้ .loc[mask, คอลัมน์] เพื่อเขียนทับค่าเฉพาะจุดที่เลือกไว้
df.loc[mask, 'จำนวนผู้มีสิทธิมาเลือกตั้ง'] = (
    df[mask].groupby('type')['จำนวนผู้มีสิทธิมาเลือกตั้ง']
    .transform(lambda x: x.values[::-1])
)

# 5. รีเซ็ต Index เพื่อความเรียบร้อย
df = df.reset_index(drop=True)

print("สลับค่าเฉพาะตำบลทุ่งฮั้ว เรียบร้อยแล้ว!")

สลับค่าเฉพาะตำบลทุ่งฮั้ว เรียบร้อยแล้ว!


In [31]:
cols_to_check = ['จำนวนผู้มีสิทธิมาเลือกตั้ง', 'จำนวนผู้มาแสดงตน']
for col in cols_to_check:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1)

df_khet = df[df['type'] == 'เขต'].copy()
df_list = df[df['type'] == 'บช'].copy()

# 3. นำ 2 ตารางมา Join กันด้วย Key สถานที่ (ตำบล และ หน่วยที่)
# suffixes จะช่วยเติมท้ายชื่อคอลัมน์ที่ซ้ำกันให้รู้ว่ามาจากตารางไหน
merged_df = pd.merge(
    df_khet, 
    df_list, 
    on=['province', 'district', 'sub-district', 'unit_number'], 
    how='inner', 
    suffixes=('_เขต', '_บช')
)

# 4. สร้างเงื่อนไขตรวจสอบ (Mismatch Conditions)
# เงื่อนไข 1: จำนวนผู้มีสิทธิฯ ไม่เท่ากัน
cond_eligible = merged_df['จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต'] != merged_df['จำนวนผู้มีสิทธิมาเลือกตั้ง_บช']

# เงื่อนไข 2: จำนวนผู้มาแสดงตน ไม่เท่ากัน
cond_voted = merged_df['จำนวนผู้มาแสดงตน_เขต'] != merged_df['จำนวนผู้มาแสดงตน_บช']

# 5. กรองเอาเฉพาะแถวที่ผิดปกติ (ตรงเงื่อนไข 1 หรือ 2 อย่างใดอย่างหนึ่ง)
error_df = merged_df[cond_eligible | cond_voted].copy()

# แสดงผลลัพธ์
if error_df.empty:
    print("✅ ข้อมูลสมบูรณ์! ยอดผู้มีสิทธิและผู้มาแสดงตนของ 'เขต' และ 'บช' ตรงกันทุกหน่วย")
else:
    print(f"❌ พบหน่วยที่ข้อมูลไม่ตรงกันจำนวน {len(error_df)} หน่วย ดังนี้:")
    
    # เลือกเฉพาะคอลัมน์ที่จำเป็นมาดูเพื่อความสะอาดตา
    display_cols = [
        'district', 'sub-district', 'unit_number', 
        'จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต', 'จำนวนผู้มีสิทธิมาเลือกตั้ง_บช',
        'จำนวนผู้มาแสดงตน_เขต', 'จำนวนผู้มาแสดงตน_บช'
    ]
    

❌ พบหน่วยที่ข้อมูลไม่ตรงกันจำนวน 193 หน่วย ดังนี้:


In [32]:
error_df

,type_เขต,province,district,sub-district,unit_number,จำนวนผู้มีสิทธิมาเลือกตั้ง_เขต,จำนวนผู้มาแสดงตน_เขต,type_บช,จำนวนผู้มีสิทธิมาเลือกตั้ง_บช,จำนวนผู้มาแสดงตน_บช
3,เขต,ลำปาง,งาว,นาแก,4,463,329,บช,468,329
5,เขต,ลำปาง,งาว,นาแก,6,277,192,บช,297,192
6,เขต,ลำปาง,งาว,บ้านร้อง,1,440,316,บช,442,316
7,เขต,ลำปาง,งาว,บ้านร้อง,2,555,345,บช,535,345
8,เขต,ลำปาง,งาว,บ้านร้อง,3,560,334,บช,50,334
...,...,...,...,...,...,...,...,...,...,...
332,เขต,ลำปาง,แจ้ห่ม,แจ้ห่ม(นอกเขต),4,469,357,บช,460,357
333,เขต,ลำปาง,แจ้ห่ม,แจ้ห่ม(นอกเขต),5,479,366,บช,515,366
340,เขต,ลำปาง,แจ้ห่ม,แจ้ห่ม(ในเขต),4,704,200,บช,304,200
349,เขต,ลำปาง,แจ้ห่ม,แม่สุก,7,421,287,บช,420,287


In [33]:
cols_to_check = ['จำนวนผู้มีสิทธิมาเลือกตั้ง', 'จำนวนผู้มาแสดงตน']
for col in cols_to_check:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)

# 2. หาค่า Max (ค่าที่สูงกว่า) ในแต่ละหน่วย
group_cols = ['province', 'district', 'sub-district', 'unit_number']

df_max = df.groupby(group_cols)[['จำนวนผู้มีสิทธิมาเลือกตั้ง', 'จำนวนผู้มาแสดงตน']].max().reset_index()
df_max.rename(columns={
    'จำนวนผู้มีสิทธิมาเลือกตั้ง': 'max_eligible',
    'จำนวนผู้มาแสดงตน': 'max_voted'
}, inplace=True)

# 3. นำค่า Max กลับไปแปะใน DataFrame หลัก
df = df.merge(df_max, on=group_cols, how='left')

# 4. สร้างเงื่อนไขที่ต้อง Manual Check
cond_manual = (df['max_eligible'] < df['max_voted']) | (df['max_eligible'] < 0)

# 5. แยก DataFrame
df_manual = df[cond_manual].copy()
cond_valid = ~cond_manual

# 6. อัปเดตกลุ่มที่ปกติ
df.loc[cond_valid, 'จำนวนผู้มีสิทธิมาเลือกตั้ง'] = df.loc[cond_valid, 'max_eligible']

# 7. ทำความสะอาด
df.drop(columns=['max_eligible', 'max_voted'], inplace=True)

# --- ส่วนการแสดงผล (ปรับใหม่) ---
if df_manual.empty:
    print("✅ ซ่อมข้อมูลอัตโนมัติสำเร็จ! ไม่มีหน่วยไหนที่ต้อง Manual Check")
else:
    # นับจำนวนหน่วยที่ผิดปกติ (ไม่ซ้ำ)
    unique_units_count = len(df_manual.drop_duplicates(subset=group_cols))
    
    print(f"⚠️ พบ {unique_units_count} หน่วย ที่ต้อง Manual Check:")
    print("   (ข้อมูลของทั้ง 'เขต' และ 'บช' มีปัญหาน้อยกว่าจำนวนผู้มาแสดงตน หรือติดลบ)\n")
    
    # เลือกคอลัมน์ที่ต้องการโชว์
    display_cols = ['type', 'district', 'sub-district', 'unit_number', 
                    'จำนวนผู้มีสิทธิมาเลือกตั้ง', 'จำนวนผู้มาแสดงตน']
    
    # เรียงลำดับเพื่อให้ เขต และ บช ของหน่วยเดียวกันอยู่ติดกัน ดูง่ายขึ้น
    df_manual_sorted = df_manual.sort_values(by=['district', 'sub-district', 'unit_number', 'type'])
    
    # พิมพ์แบบมี Index=False เพื่อความสะอาด
    print(df_manual_sorted[display_cols].to_string(index=False))

⚠️ พบ 30 หน่วย ที่ต้อง Manual Check:
   (ข้อมูลของทั้ง 'เขต' และ 'บช' มีปัญหาน้อยกว่าจำนวนผู้มาแสดงตน หรือติดลบ)

type   district sub-district  unit_number  จำนวนผู้มีสิทธิมาเลือกตั้ง  จำนวนผู้มาแสดงตน
  บช        งาว         นาแก            3                          37               244
 เขต        งาว         นาแก            3                          37               244
  บช        งาว     บ้านร้อง            4                          40               254
 เขต        งาว     บ้านร้อง            4                          40               254
  บช        งาว     บ้านโป่ง            6                          -1               283
 เขต        งาว     บ้านโป่ง            6                          23               283
  บช        งาว      หลวงใต้            4                          70               223
 เขต        งาว      หลวงใต้            4                          -1               223
  บช        งาว       แม่ตีบ            2                          57               342
 เขต  

In [34]:
df.to_csv("../master_totalpeople.csv")

Mannaul On above unit